# IMAS Reader Basics

This notebook is for a downstream user who wants to read SOLEDGE-HDG IMAS `.nc` files and quickly inspect what was saved.

It uses the small helper functions now provided in `hdg_postprocess.imas_export.reader`.

It follows the three export patterns used in the IMAS export tutorial:

- one steady-state case per file
- one bundled scan with one IDS occurrence per simulation
- one full discharge with multiple time indices in one file


In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from matplotlib.colors import LogNorm, Normalize

from hdg_postprocess.imas_export.reader import (
    equilibrium_slice,
    field_style_presets,
    load_ids,
    make_field_animation,
    plasma_slice,
    plot_field_2d,
    save_animation,
    symmetric_limits,
)


## Small helpers

These helpers keep the examples below compact and make the plotting choices explicit.


In [ ]:
styles = field_style_presets()


def positive_log_norm(field):
    values = np.asarray(field, dtype=float)
    finite = values[np.isfinite(values) & (values > 0)]
    if finite.size == 0:
        return None
    return LogNorm(vmin=float(finite.min()), vmax=float(finite.max()))


def frame_norm(field_name, frames):
    style = styles[field_name]
    if style["scale"] == "log":
        return positive_log_norm(np.asarray(frames, dtype=float))
    values = np.asarray(frames, dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return None
    if style.get("symmetric"):
        vmin, vmax = symmetric_limits(finite)
        return Normalize(vmin=vmin, vmax=vmax)
    return Normalize(vmin=float(finite.min()), vmax=float(finite.max()))


def plot_with_colorbar(r, z, field, *, title, label, cmap, norm=None, separatrix=None):
    fig, ax = plt.subplots(figsize=(6, 8))
    mesh, color_label = plot_field_2d(
        ax,
        r=r,
        z=z,
        field=field,
        title=title,
        label=label,
        cmap=cmap,
        norm=norm,
        separatrix=separatrix,
    )
    fig.colorbar(mesh, ax=ax, label=color_label)
    plt.show()


def separatrix_overlay(eq_slice, level=None):
    if level is None:
        return None
    return {"psi": eq_slice["psi"], "level": float(level)}


## 1. Read one steady-state case

A single exported case lives at IDS occurrence `0` and time index `0`.


In [ ]:
steady_db_path = "path/to/imas_single_case.nc"
summary, equilibrium, plasma = load_ids(steady_db_path, occurrence=0)

summary_params = json.loads(str(summary.code.parameters))
eq = equilibrium_slice(equilibrium, plasma, time_index=0)
pl = plasma_slice(plasma, time_index=0)

print("Description:", summary.description)
print("Workflow:", summary.simulation.workflow)
print("Puff rate:", summary_params.get("puff_rate"))
print("Recycling:", summary_params.get("recycling_coefficient"))
print("Zeff:", pl["zeff"])


In [ ]:
separatrix_level = None  # Set the known separatrix psi level here if you have one.
sep = separatrix_overlay(eq, separatrix_level)

plot_with_colorbar(
    eq["r"],
    eq["z"],
    eq["psi"],
    title="Steady case: poloidal flux",
    label="psi",
    cmap=styles["psi"]["cmap"],
    separatrix=sep,
)

fig, ax = plt.subplots(figsize=(6, 8))
contours = ax.contour(eq["r"], eq["z"], eq["psi"], levels=18, cmap=styles["psi"]["cmap"])
if sep is not None:
    ax.contour(eq["r"], eq["z"], eq["psi"], levels=[sep["level"]], colors="black", linewidths=1.2)
fig.colorbar(contours, ax=ax, label="psi")
ax.set_aspect("equal")
ax.set_xlabel("R [m]")
ax.set_ylabel("Z [m]")
ax.set_title("Steady case: psi contours")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
for ax, cmap in zip(axes, ["inferno", "magma", "cividis"]):
    mesh, _ = plot_field_2d(
        ax,
        r=eq["r"],
        z=eq["z"],
        field=pl["ne"],
        title=f"n_e with {cmap}",
        label="n_e",
        cmap=cmap,
        norm=positive_log_norm(pl["ne"]),
        separatrix=sep,
    )
    fig.colorbar(mesh, ax=ax)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
vmin, vmax = symmetric_limits(eq["br"])
for ax, cmap in zip(axes, ["bwr", "RdBu_r", "coolwarm"]):
    mesh, _ = plot_field_2d(
        ax,
        r=eq["r"],
        z=eq["z"],
        field=eq["br"],
        title=f"B_R with {cmap}",
        label="B_R [T]",
        cmap=cmap,
        norm=Normalize(vmin=vmin, vmax=vmax),
        separatrix=sep,
    )
    fig.colorbar(mesh, ax=ax)
plt.show()


## 2. Read one simulation from a bundled scan

For the bundled scan export, the scan point is selected by IDS occurrence.


In [ ]:
scan_db_path = "path/to/puff_scan.nc"
occurrence = 1

summary, equilibrium, plasma = load_ids(scan_db_path, occurrence=occurrence)
summary_params = json.loads(str(summary.code.parameters))
eq = equilibrium_slice(equilibrium, plasma, time_index=0)
pl = plasma_slice(plasma, time_index=0)
sep = separatrix_overlay(eq, separatrix_level)

print("Occurrence:", occurrence)
print("Description:", summary.description)
print("Stored export run metadata:", summary_params.get("export_run"))
print("Puff rate:", summary_params.get("puff_rate"))

plot_with_colorbar(
    eq["r"],
    eq["z"],
    pl["te"],
    title=f"Bundled scan occurrence {occurrence}: electron temperature",
    label="T_e [eV]",
    cmap=styles["te"]["cmap"],
    norm=positive_log_norm(pl["te"]),
    separatrix=sep,
)
plot_with_colorbar(
    eq["r"],
    eq["z"],
    pl["nn"],
    title=f"Bundled scan occurrence {occurrence}: neutral density",
    label="n_n [m^-3]",
    cmap=styles["nn"]["cmap"],
    norm=positive_log_norm(pl["nn"]),
    separatrix=sep,
)


## 3. Read one time index from a full discharge

For the full-discharge export, the IDS occurrence stays fixed and the snapshot is selected by `time_index`.


In [ ]:
discharge_db_path = "path/to/full_discharge.nc"
occurrence = 0
time_index = 0

summary, equilibrium, plasma = load_ids(discharge_db_path, occurrence=occurrence)
summary_params = json.loads(str(summary.code.parameters))
eq = equilibrium_slice(equilibrium, plasma, time_index=time_index)
pl = plasma_slice(plasma, time_index=time_index)
sep = separatrix_overlay(eq, separatrix_level)

print("Description:", summary.description)
print("Workflow:", summary.simulation.workflow)
print("Number of exported snapshots:", len(equilibrium.time))
print("Selected time [s]:", eq["time"])
print("All exported times [s]:", np.asarray(equilibrium.time))


In [ ]:
plot_with_colorbar(
    eq["r"],
    eq["z"],
    pl["ti"],
    title=f"Full discharge time index {time_index}: ion temperature",
    label="T_i [eV]",
    cmap=styles["ti"]["cmap"],
    norm=positive_log_norm(pl["ti"]),
    separatrix=sep,
)

vmin, vmax = symmetric_limits(eq["bz"])
plot_with_colorbar(
    eq["r"],
    eq["z"],
    eq["bz"],
    title=f"Full discharge time index {time_index}: B_Z",
    label="B_Z [T]",
    cmap=styles["bz"]["cmap"],
    norm=Normalize(vmin=vmin, vmax=vmax),
    separatrix=sep,
)


## 4. Animate saved fields

The helper below can animate either a repeated grid or a time-varying rectangular grid. Set `separatrix_level` above if you know the correct contour level for your export.


In [ ]:
times = np.asarray(plasma.time, dtype=float)
all_eq = [equilibrium_slice(equilibrium, plasma, time_index=i) for i in range(len(times))]
all_plasma = [plasma_slice(plasma, time_index=i) for i in range(len(times))]

saved_frames = {
    "ne": [frame["ne"] for frame in all_plasma],
    "te": [frame["te"] for frame in all_plasma],
    "ti": [frame["ti"] for frame in all_plasma],
    "u_par": [frame["u_par"] for frame in all_plasma],
    "nn": [frame["nn"] for frame in all_plasma],
    "psi": [frame["psi"] for frame in all_eq],
    "br": [frame["br"] for frame in all_eq],
    "bz": [frame["bz"] for frame in all_eq],
    "bphi": [frame["bphi"] for frame in all_eq],
}

field_name = "te"  # Try: ne, te, ti, u_par, nn, psi, br, bz, bphi
frames = saved_frames[field_name]
style = styles[field_name]
r_frames = [frame["r"] for frame in all_eq]
z_frames = [frame["z"] for frame in all_eq]
sep_frames = None if separatrix_level is None else [
    {"psi": frame["psi"], "level": float(separatrix_level)} for frame in all_eq
]

fig, animation = make_field_animation(
    frames=frames,
    times=times,
    title=f"{field_name} animation",
    label=field_name,
    cmap=style["cmap"],
    norm=frame_norm(field_name, frames),
    separatrix_frames=sep_frames,
    r_frames=r_frames,
    z_frames=z_frames,
)
plt.close(fig)
HTML(animation.to_jshtml())

# To save an MP4 after you have ffmpeg available:
# save_animation(animation, f"{field_name}_discharge.mp4", fps=8, dpi=140)


## Practical notes

- Outside the original HDG mesh, the exporter writes `NaN`.
- `equilibrium` does not store `R` and `Z` as data arrays; reconstruct coordinates from the referenced GGD topology.
- In full-discharge files, `plasma_profiles.grid_ggd[time_index]` may itself be an IMAS `path` reference to an earlier explicit grid entry, and the reader helpers resolve that automatically.
- In the current single-ion export, `electrons.density` is the authoritative density field; the redundant `ion[0].density` and `n_i_total` fields are intentionally left empty.
- If `Zeff` is spatially constant, it is stored in `plasma_profiles.global_quantities.z_eff_resistive` instead of as a 2D field.
- Good starting colormaps are:
  - log plasma fields (`n_e`, `n_n`, `T_e`, `T_i`): `inferno`, `magma`, `cividis`
  - signed fields (`u_par`, `B_R`, `B_Z`, often also `M` when available): `bwr`
  - monotonic equilibrium scalars (`psi`, `B_phi`): `cividis` or `viridis`
- If you want one fully validated export before production runs, keep IMAS validation enabled for a small smoke test, read the file back, and compare key fields against direct HDG sampling.
